# RAG-система для пошуку відповідей у документах

## 1. Мета та предметна область
**Мета:** створити інформаційно-пошукову систему (RAG), яка знаходить потрібні фрагменти у базі PDF-документів і генерує точну відповідь із посиланням на джерело.
**База знань:** корпус офіційних нормативних PDF-документів Державного університету «Житомирська політехніка» (5 документів, ~9.45 МБ).

## 2. Архітектура системи
1. **Підготовка даних:** витягування тексту з PDF (`pypdf`), очищення від сміття, розбиття на чанки по 1000 символів із перекриттям 200.
2. **Індексація:** створення векторного індексу (`FAISS` + `Qwen3-Embedding-0.6B`) та лексичного індексу (`BM25`).
3. **Пошук (Retrieval):** гібридний пошук (об'єднання результатів FAISS та BM25 через RRF).
4. **Генерація:** формування відповіді моделлю `Qwen3-4B-Instruct-2507` суворо за знайденим контекстом.

In [1]:
!pip install -q langchain langchain-core langchain-community langchain-huggingface langchain-text-splitters pypdf faiss-cpu rank_bm25 sentence-transformers transformers accelerate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os
import re
from pathlib import Path

# Папка, куди потрібно покласти PDF-документи
DOCS_DIR = Path("./data_pdfs")
DOCS_DIR.mkdir(exist_ok=True)

# Параметри розбиття тексту (Chunking)
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Назви моделей
EMBEDDING_MODEL_NAME = "Qwen/Qwen3-Embedding-0.6B"
LLM_MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"

print(f"Папка для документів готова: {DOCS_DIR.resolve()}")
print(f"Знайдено PDF-файлів: {len(list(DOCS_DIR.glob('*.pdf')))}")

Папка для документів готова: /content/data_pdfs
Знайдено PDF-файлів: 5


In [4]:
from pypdf import PdfReader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []

for pdf_path in sorted(DOCS_DIR.glob("*.pdf")):
    reader = PdfReader(pdf_path)
    full_text = []

    for page in reader.pages:
        text = page.extract_text() or ""
        # Відсіюємо порожні сторінки або скани без тексту
        if len(text.strip()) >= 100:
            full_text.append(text.strip())

    doc_text = "\n\n".join(full_text)
    if not doc_text:
        print(f"Пропущено (немає тексту): {pdf_path.name}")
        continue

    # Розбиваємо документ на фрагменти
    raw_chunks = splitter.split_text(doc_text)
    for i, chunk_text in enumerate(raw_chunks):
        # Додаємо назву файлу на початок чанка, щоб модель не втрачала контекст
        enriched_text = f"Документ: {pdf_path.stem}\n{chunk_text}"
        chunks.append(
            Document(
                page_content=enriched_text,
                metadata={"source": pdf_path.name, "chunk_id": i}
            )
        )

print(f"✅ Готово! Створено фрагментів (чанків): {len(chunks)}")
if chunks:
    print("\nПриклад першого чанка:\n" + "-"*40)
    print(chunks[0].page_content[:400] + "...")

✅ Готово! Створено фрагментів (чанків): 733

Приклад першого чанка:
----------------------------------------
Документ: kolektyvnyj dohovir
КОЛЕКТИВНИЙ ДОГОВІР 
 
 
між адміністрацією та профспілковим 
комітетом викладачів та співробітників 
Державного університету «Житомирська 
політехніка» на 2022–2026 роки

КОЛЕКТИВНИЙ ДОГОВІР 
 
 
між адміністрацією та профспілковим 
комітетом викладачів та співробітників 
Державного університету «Житомирська 
політехніка» на 2022–2026 роки 
 
 
 
Схвалено на Конферен...


### Створення індексів (FAISS + BM25)

In [5]:
import torch
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.vectorstores.utils import DistanceStrategy
from rank_bm25 import BM25Okapi

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Використовується пристрій: {device}")

# 1. Векторний індекс (FAISS)
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True}
)

vectorstore = FAISS.from_documents(
    chunks,
    embeddings,
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT
)

# 2. Лексичний індекс (BM25) для точного пошуку слів
def tokenize(text):
    return re.findall(r"\w+", text.lower())

tokenized_corpus = [tokenize(doc.page_content) for doc in chunks]
bm25 = BM25Okapi(tokenized_corpus)

print("✅ Обидва індекси (FAISS та BM25) успішно створено!")

/tmp/ipykernel_785/3504730643.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


Використовується пристрій: cuda


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

✅ Обидва індекси (FAISS та BM25) успішно створено!


### Гібридний пошук (RRF) і тест

In [6]:
def hybrid_search(query: str, top_k: int = 5, k_rrf: int = 60):
    # 1. Пошук через FAISS (семантика)
    faiss_docs = vectorstore.similarity_search(query, k=top_k * 2)

    # 2. Пошук через BM25 (точні слова)
    scores = bm25.get_scores(tokenize(query))
    top_bm25_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:top_k * 2]
    bm25_docs = [chunks[i] for i in top_bm25_idx]

    # 3. Об'єднання результатів (RRF)
    rrf_scores = {}
    doc_map = {}

    for rank, doc in enumerate(faiss_docs, start=1):
        key = (doc.metadata["source"], doc.metadata["chunk_id"])
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (k_rrf + rank)
        doc_map[key] = doc

    for rank, doc in enumerate(bm25_docs, start=1):
        key = (doc.metadata["source"], doc.metadata["chunk_id"])
        rrf_scores[key] = rrf_scores.get(key, 0.0) + 1.0 / (k_rrf + rank)
        doc_map[key] = doc

    # Сортуємо за підсумковим балом
    ranked_keys = sorted(rrf_scores.keys(), key=lambda x: rrf_scores[x], reverse=True)[:top_k]
    return [doc_map[k] for k in ranked_keys]

# Тестуємо пошук
test_query = "Яка мета студентського самоврядування університету?"
results = hybrid_search(test_query, top_k=3)

print(f"🔍 Запит: {test_query}\n" + "="*50)
for i, doc in enumerate(results, 1):
    print(f"[{i}] Джерело: {doc.metadata['source']} (чанк {doc.metadata['chunk_id']})")
    print(doc.page_content[:250].replace("\n", " ") + "...\n")

🔍 Запит: Яка мета студентського самоврядування університету?
[1] Джерело: polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf (чанк 7)
Документ: polozhennja_pro_studentske_samovrjaduvannja_06_2024 університету;  • отримувати приміщення та техніку від адміністрації для  здійснення своїх повноважень;  • вільно формувати та висловлювати свою думку з усіх питань  діяльності університету...

[2] Джерело: polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf (чанк 1)
Документ: polozhennja_pro_studentske_samovrjaduvannja_06_2024 Житомирська  політехніка  МІНІСТЕРСТВО ОСВІТИ І НАУКИ УКРАЇНИ  ДЕРЖАВНИЙ УНІВЕРСИТЕТ «ЖИТОМИРСЬКА ПОЛІТЕХНІКА»  Система управління якістю відповідає ДСТУ ISO 9001:2015  П-03.03- 01.01-04- ...

[3] Джерело: statut_zhytomyrska_politehnika_2022_z_pidpysamy.pdf (чанк 91)
Документ: statut_zhytomyrska_politehnika_2022_z_pidpysamy самоврядування, яке є невід’ємною частиною громадського самоврядування  Університету. Студентське самоврядування – це право і можливість сту

### Завантаження генеративної моделі (LLM)

In [7]:
from transformers import AutoModelForCausalLM, AutoTokenizer

print(f"Завантаження моделі {LLM_MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("✅ Мовна модель успішно завантажена і готова до роботи!")

Завантаження моделі Qwen/Qwen3-4B-Instruct-2507...


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.38k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

... (4 outputs left out)

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

✅ Мовна модель успішно завантажена і готова до роботи!


### Повний RAG-пайплайн та тестування

In [8]:
SYSTEM_PROMPT = """Ти — точний асистент, який відповідає на запитання виключно на основі наданих фрагментів документів.
Правила:
1. Використовуй ТІЛЬКИ інформацію з наданого контексту. Не вигадуй факти.
2. Після кожного твердження став посилання на номер фрагмента у квадратних дужках, наприклад [1] або [1, 3].
3. Якщо в контексті немає відповіді на запитання, чітко напиши: «У наданих документах немає інформації для відповіді на це запитання».
4. Відповідай українською мовою, лаконічно і по суті."""

def ask_rag(query: str, top_k: int = 5):
    # 1. Гібридний пошук
    docs = hybrid_search(query, top_k=top_k)

    # 2. Формування пронумерованого контексту
    context_parts = []
    for i, doc in enumerate(docs, 1):
        context_parts.append(f"[{i}] (Файл: {doc.metadata['source']})\n{doc.page_content}")
    context = "\n\n---\n\n".join(context_parts)

    # 3. Підготовка повідомлень для моделі
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Контекст:\n{context}\n\nЗапитання: {query}"}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # 4. Генерація (детермінований режим do_sample=False)
    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            do_sample=False
        )

    # Відрізаємо вхідний промпт, залишаємо тільки нову відповідь
    output_ids = generated_ids[0][inputs.input_ids.shape[1]:]
    answer = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    # Вивід результату
    print(f"ЗАПИТАННЯ: {query}\n")
    print(f"ВІДПОВІДЬ:\n{answer}\n")
    print("ВИКОРИСТАНІ ДЖЕРЕЛА:")
    for i, doc in enumerate(docs, 1):
        print(f"  [{i}] {doc.metadata['source']} (чанк {doc.metadata['chunk_id']})")
    print("=" * 60)

# Тест 1: питання, відповідь на яке є в документах
ask_rag("Яка мета студентського самоврядування університету?")

# Тест 2: перевірка на відмову (питання не з документів)
ask_rag("Скільки коштує квиток на літак до Марса?")

ЗАПИТАННЯ: Яка мета студентського самоврядування університету?

ВІДПОВІДЬ:
Метою студентського самоврядування університету є об’єднання студентів для вирішення питань навчання і побуту, захисту прав та інтересів студентів, а також участі в управлінні університетом [4, 5].

ВИКОРИСТАНІ ДЖЕРЕЛА:
  [1] polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf (чанк 7)
  [2] polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf (чанк 1)
  [3] polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf (чанк 34)
  [4] statut_zhytomyrska_politehnika_2022_z_pidpysamy.pdf (чанк 91)
  [5] polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf (чанк 3)
ЗАПИТАННЯ: Скільки коштує квиток на літак до Марса?

ВІДПОВІДЬ:
У наданих документах немає інформації для відповіді на це запитання.

ВИКОРИСТАНІ ДЖЕРЕЛА:
  [1] polozhennja_pro_orhanizaciju_osvitnoho_procesu (7).pdf (чанк 264)
  [2] kolektyvnyj dohovir.pdf (чанк 82)
  [3] kolektyvnyj dohovir.pdf (чанк 120)
  [4] polozhennja_pro_orhanizaciju_osvitnoho

### 1.2. Бенчмарк

In [11]:
# 1.2. Бенчмарк (24 питання)
# expected_doc = ключова частина назви файлу (або None, якщо відповіді в корпусі немає)
benchmark = [
    # --- колективний договір ---
    {"q": "На які роки укладено Колективний договір Житомирської політехніки?", "doc": "kolektyvnyj", "type": "фактичне", "ref": "На 2022–2026 роки."},
    {"q": "Між ким укладено Колективний договір університету?", "doc": "kolektyvnyj", "type": "фактичне", "ref": "Між адміністрацією та профспілковим комітетом викладачів та співробітників."},
    {"q": "На кого поширюється дія Колективного договору?", "doc": "kolektyvnyj", "type": "процедурне", "ref": "На всіх працівників університету та адміністрацію."},
    {"q": "Як регулюється тривалість робочого часу та відпусток працівників університету?", "doc": "kolektyvnyj", "type": "детальне", "ref": "Визначається Колективним договором відповідно до законодавства про працю."},

    # --- положення про вчену раду ---
    {"q": "Який статус має Вчена рада університету?", "doc": "vchenu radu", "type": "фактичне", "ref": "Колегіальний орган управління університету."},
    {"q": "Хто очолює Вчену раду Житомирської політехніки?", "doc": "vchenu radu", "type": "фактичне", "ref": "Голова Вченої ради, який обирається/призначається відповідно до положення."},
    {"q": "Які основні повноваження має Вчена рада університету?", "doc": "vchenu radu", "type": "процедурне", "ref": "Визначення стратегії розвитку, затвердження положень, обрання на посади тощо."},
    {"q": "Як часто проводяться засідання Вченої ради та за яких умов вони є правомочними?", "doc": "vchenu radu", "type": "детальне", "ref": "За наявності кворуму (більшості від затвердженого складу)."},

    # --- положення про організацію освітнього процесу ---
    {"q": "Які форми здобуття вищої освіти існують в університеті?", "doc": "orhanizaciju_osvitnoho_procesu", "type": "фактичне", "ref": "Очна (денна, вечірня), заочна, дистанційна, дуальна тощо."},
    {"q": "Як здійснюється оцінювання знань студентів та яка шкала використовується?", "doc": "orhanizaciju_osvitnoho_procesu", "type": "детальне", "ref": "За 100-бальною шкалою, шкалою ЄКТС та національною шкалою."},
    {"q": "Який порядок ліквідації академічної заборгованості студентами?", "doc": "orhanizaciju_osvitnoho_procesu", "type": "процедурне", "ref": "Визначається графіком перескладань у встановлені терміни."},
    {"q": "З яких елементів складається навчальний час студента (кредити ЄКТС)?", "doc": "orhanizaciju_osvitnoho_procesu", "type": "фактичне", "ref": "Один кредит ЄКТС становить 30 годин навчального навантаження."},

    # --- положення про студентське самоврядування ---
    {"q": "Яка мета студентського самоврядування університету?", "doc": "studentske_samovrjaduvannja", "type": "фактичне", "ref": "Вирішення питань навчання і побуту, захист прав та інтересів студентів, участь в управлінні."},
    {"q": "Хто може бути обраним до органів студентського самоврядування?", "doc": "studentske_samovrjaduvannja", "type": "процедурне", "ref": "Особи, які навчаються в університеті (студенти)."},
    {"q": "Які права мають органи студентського самоврядування щодо приміщень і техніки?", "doc": "studentske_samovrjaduvannja", "type": "детальне", "ref": "Отримувати приміщення та техніку від адміністрації для здійснення повноважень."},
    {"q": "Як фінансується діяльність студентського самоврядування?", "doc": "studentske_samovrjaduvannja", "type": "детальне", "ref": "За рахунок коштів університету (не менше встановленого відсотка власних надходжень) та інших джерел."},

    # --- статут ---
    {"q": "Яке повне офіційне найменування університету згідно зі Статутом?", "doc": "statut", "type": "фактичне", "ref": "Державний університет «Житомирська політехніка»."},
    {"q": "Хто є засновником Державного університету «Житомирська політехніка»?", "doc": "statut", "type": "фактичне", "ref": "Держава в особі Міністерства освіти і науки України."},
    {"q": "Які органи управління та громадського самоврядування діють в університеті за Статутом?", "doc": "statut", "type": "процедурне", "ref": "Ректор, Вчена рада, Конференція трудового колективу, органи студентського самоврядування тощо."},
    {"q": "Яка юридична адреса університету вказана у Статуті?", "doc": "statut", "type": "детальне", "ref": "м. Житомир, вул. Чуднівська, 103."},

    # --- питання БЕЗ відповіді в корпусі (перевірка відмов) ---
    {"q": "Скільки коштує проживання в гуртожитку №3 у 2026 році?", "doc": None, "type": "без відповіді", "ref": "У наданих документах немає інформації."},
    {"q": "Який розмір мінімальної академічної стипендії у гривнях?", "doc": None, "type": "без відповіді", "ref": "У наданих документах немає інформації."},
    {"q": "Який розклад дзвінків у Житомирській політехніці?", "doc": None, "type": "без відповіді", "ref": "У наданих документах немає інформації."},
    {"q": "Хто переміг у чемпіонаті світу з футболу 2022 року?", "doc": None, "type": "без відповіді", "ref": "У наданих документах немає інформації."}
]

print(f"Створено бенчмарк: {len(benchmark)} питань (із відповіддю: {sum(1 for x in benchmark if x['doc'])}, без відповіді: {sum(1 for x in benchmark if not x['doc'])})")

Створено бенчмарк: 24 питань (із відповіддю: 20, без відповіді: 4)


### 1.4. Оцінка ретриву: порівняння 2 конфігурацій (FAISS vs Гібридний RRF)

In [12]:
def faiss_only_search(query: str, top_k: int = 5):
    return vectorstore.similarity_search(query, k=top_k)

def eval_retriever(search_fn, dataset, top_k=5):
    answerable = [item for item in dataset if item["doc"] is not None]
    hits = 0
    mrr_total = 0.0
    missed = []

    for item in answerable:
        docs = search_fn(item["q"], top_k=top_k)
        found_rank = None
        for rank, d in enumerate(docs, start=1):
            if item["doc"].lower() in d.metadata["source"].lower():
                found_rank = rank
                break
        if found_rank is not None:
            hits += 1
            mrr_total += 1.0 / found_rank
        else:
            missed.append((item["q"], item["doc"], [d.metadata["source"] for d in docs[:2]]))

    return hits / len(answerable), mrr_total / len(answerable), missed

hr_faiss, mrr_faiss, miss_faiss = eval_retriever(faiss_only_search, benchmark, top_k=5)
hr_hybrid, mrr_hybrid, miss_hybrid = eval_retriever(hybrid_search, benchmark, top_k=5)

print("ПОРІВНЯННЯ КОНФІГУРАЦІЙ РЕТРИВУ (top_k=5)")
print("=" * 55)
print(f"{'Конфігурація':<30} | {'Hit Rate@5':<10} | {'MRR@5':<8}")
print("-" * 55)
print(f"{'1. Тільки FAISS (семантика)':<30} | {hr_faiss:<10.2%} | {mrr_faiss:<8.3f}")
print(f"{'2. Гібридний (FAISS + BM25)':<30} | {hr_hybrid:<10.2%} | {mrr_hybrid:<8.3f}")
print("=" * 55)

if miss_hybrid:
    print(f"\nПромахи гібридного ретриву ({len(miss_hybrid)}):")
    for q, exp, got in miss_hybrid:
        print(f" - Питання: {q}\n   Очікувалось: {exp} | Отримано: {got}")

ПОРІВНЯННЯ КОНФІГУРАЦІЙ РЕТРИВУ (top_k=5)
Конфігурація                   | Hit Rate@5 | MRR@5   
-------------------------------------------------------
1. Тільки FAISS (семантика)    | 100.00%    | 0.900   
2. Гібридний (FAISS + BM25)    | 100.00%    | 0.887   


### 1.5. Прогін генерації на всьому бенчмарку

In [13]:
import re

def rag_answer_silent(query: str, top_k: int = 5):
    docs = hybrid_search(query, top_k=top_k)
    context = "\n\n---\n\n".join(
        f"[{i}] (Файл: {d.metadata['source']})\n{d.page_content}" for i, d in enumerate(docs, 1)
    )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Контекст:\n{context}\n\nЗапитання: {query}"}
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        gen_ids = model.generate(**inputs, max_new_tokens=256, do_sample=False)
    ans = tokenizer.decode(gen_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
    return ans, [d.metadata["source"] for d in docs]

results_log = []
has_citation_count = 0
refusal_correct = 0
refusal_total = 0

for idx, item in enumerate(benchmark, 1):
    ans, sources = rag_answer_silent(item["q"], top_k=5)
    is_refusal = "немає інформації" in ans.lower()
    has_cite = bool(re.search(r"\[\d+", ans))

    if item["doc"] is None:
        refusal_total += 1
        if is_refusal:
            refusal_correct += 1
    else:
        if has_cite and not is_refusal:
            has_citation_count += 1

    results_log.append({**item, "answer": ans, "sources": sources, "is_refusal": is_refusal, "has_cite": has_cite})
    print(f"[{idx:02d}] ({item['type']}) {item['q']}")
    print(f"     🤖 {ans[:220]}")
    print(f"     📎 Цитування: {'✅' if has_cite else '❌'} | Відмова: {'Так' if is_refusal else 'Ні'}\n")

print("=" * 60)
print(f"Відповіді з цитуванням (на питання з бази): {has_citation_count} / {len(benchmark) - refusal_total}")
print(f"Коректні відмови (на питання поза базою):   {refusal_correct} / {refusal_total}")

[01] (фактичне) На які роки укладено Колективний договір Житомирської політехніки?
     🤖 Колективний договір Житомирської політехніки укладено на 2022–2026 роки [5].
     📎 Цитування: ✅ | Відмова: Ні

[02] (фактичне) Між ким укладено Колективний договір університету?
     🤖 Колективний договір укладено між ректором як представником держави, що є власником Державного університету «Житомирська політехніка», і найманими працівниками університету, в особі голови первинної профспілкової організа
     📎 Цитування: ✅ | Відмова: Ні

[03] (процедурне) На кого поширюється дія Колективного договору?
     🤖 Дія Колективного договору поширюється на трудовий колектив Житомирської політехніки, а саме на працівників та здобувачів освіти [1, 5]. Профком представляє інтереси трудового колективу у галузі освіти, праці, побуту, кул
     📎 Цитування: ✅ | Відмова: Ні

[04] (детальне) Як регулюється тривалість робочого часу та відпусток працівників університету?
     🤖 Тривалість робочого часу та відпусток 

### 1.7. Демонстрація

In [14]:
!pip install -q gradio
import gradio as gr

def gradio_rag(question):
    if not question.strip():
        return "Введіть запитання.", ""
    ans, sources = rag_answer_silent(question, top_k=5)
    src_text = "\n".join(f"[{i}] {s}" for i, s in enumerate(sources, 1))
    return ans, src_text

demo = gr.Interface(
    fn=gradio_rag,
    inputs=gr.Textbox(label="Запитання до документів університету", placeholder="Наприклад: Яка мета студентського самоврядування?"),
    outputs=[
        gr.Textbox(label="Відповідь RAG-системи", lines=5),
        gr.Textbox(label="Використані джерела", lines=5)
    ],
    title="RAG-асистент Житомирської політехніки",
    description="Ставте запитання щодо нормативних документів університету. Система відповідає суворо за контекстом із посиланням на джерела."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

# 1.8. Мій звіт

### 1. Корпус і чому потрібен RAG
* **Корпус:** 5 офіційних нормативних документів Державного університету «Житомирська політехніка» у форматі PDF загальним обсягом **~9.45 МБ**:
  1. `kolektyvnyj dohovir.pdf` (5572.6 КБ)
  2. `polozhennja pro vchenu radu_2026.pdf` (735.8 КБ)
  3. `polozhennja_pro_orhanizaciju_osvitnoho_procesu (7).pdf` (1363.5 КБ)
  4. `polozhennja_pro_studentske_samovrjaduvannja_06_2024.pdf` (926.8 КБ)
  5. `statut_zhytomyrska_politehnika_2022_z_pidpysamy.pdf` (1077.0 КБ)
* **Джерело та ліцензія:** офіційний сайт ДУ «Житомирська політехніка» (публічна інформація у відкритому доступі).
* **Цільова аудиторія:** студенти, викладачі та адміністрація університету, яким потрібно швидко знаходити точні норми, правила оцінювання, права чи процедури без ручного пошуку по сотнях сторінок.
* **Чому саме RAG, а не промпт чи донавчання:**
  * *Промпт (передача всіх документів у контекст)* не підходить через великий обсяг тексту (733 фрагменти перевищують ефективне вікно контексту компактних моделей і різко збільшують час генерації).
  * *Донавчання (fine-tuning)* не гарантує точного цитування першоджерел, схильне до галюцинацій і вимагає повного перенавчання моделі щоразу, коли університет оновлює положення. RAG дозволяє миттєво оновити базу, замінивши PDF-файл, і гарантує перевіреність відповіді через посилання.

---

### 2. Бенчмарк
Складено набір із **24 запитань** різних типів:
* **20 запитань із відповідями в корпусі** (по 4 на кожен документ: фактичні, процедурні та детальні).
* **4 запитання поза межами корпусу** (вартість гуртожитку, розмір стипендії, розклад дзвінків, ЧС-2022) — для перевірки здатності системи коректно відмовлятися від відповіді.

---

### 3. Індексація (модель ембедингів, розбиття, сховище)
* **Парсинг:** `pypdf` із фільтрацією порожніх/відсканованих сторінок (< 100 символів). На початок кожного фрагмента додано назву документа для збереження контексту.
* **Розбиття (Chunking):** `RecursiveCharacterTextSplitter` із параметрами `chunk_size = 1000` символів та `chunk_overlap = 200` символів. Отримано **733 фрагменти**.
* **Модель ембедингів:** `Qwen/Qwen3-Embedding-0.6B` (нормалізовані вектори, висока якість для української мови).
* **Сховище:** `FAISS` із метрикою `MAX_INNER_PRODUCT` (еквівалент косинусної подібності для нормалізованих векторів) + лексичний індекс `BM25Okapi` для гібридного пошуку.

---

### 4. Результати ретриву для різних конфігурацій
Проведено порівняння двох конфігурацій пошуку (`top_k = 5`) на 20 запитаннях бенчмарку:

| Конфігурація | Hit Rate@5 | MRR@5 |
| :--- | :---: | :---: |
| 1. Тільки FAISS (семантичний пошук) | **100.00%** | **0.900** |
| 2. Гібридний пошук (FAISS + BM25 через RRF) | **100.00%** | **0.887** |

**Висновок по ретриву:** обидві конфігурації знайшли потрібний документ у топ-5 для 100% запитань, проте чистий семантичний пошук (`FAISS`) показав дещо вищий `MRR@5` (`0.900` проти `0.887`), оскільки в нормативних текстах багато повторюваних слів («університет», «положення»), які створюють шум для `BM25` без списку стоп-слів.

---

### 5. Генеративна модель і результати відповідей
* **Модель:** `Qwen/Qwen3-4B-Instruct-2507` (локально на GPU у `float16`, детермінований режим `do_sample=False`).
* **Результати на бенчмарку:**
  * Відповіді з цитуванням на запитання з бази: **19 / 20 (95%)**.
  * Коректні відмови на запитання поза базою: **4 / 4 (100%)** — жодної галюцинації на «порожніх» запитах.

---

### 6. Аналіз відмов та проблемних випадків
1. **Питання [18] («Хто є засновником ДУ «Житомирська політехніка»?»)**
   * *Тип відмови:* хибна відмова / промах ретриву на рівні фрагмента.
   * *Причина:* Статут потрапив у топ-5, але конкретний чанк зі згадкою про МОН не потрапив у контекст, тому модель відповіла, що інформації немає.
   * *Як виправити:* збільшити `top_k` (наприклад, до 7–8) або додати етап переранжування (Cross-Encoder Reranker).
2. **Питання [20] («Яка юридична адреса університету вказана у Статуті?»)**
   * *Тип відмови:* помилка форматування цитування.
   * *Причина:* модель правильно знайшла адресу, але поставила посилання `[1.3]` замість номера фрагмента `[1]` — переплутала номер пункту Статуту (`1.3`) із номером чанка.
   * *Як виправити:* додати в системний промпт явний приклад (few-shot) і заборону використовувати номери пунктів документа як цитати.
3. **Питання [11] («Який порядок ліквідації академічної заборгованості студентами?»)**
   * *Тип відмови:* семантична підміна понять у ретриві.
   * *Причина:* ретривер вивів на перші позиції фрагмент про *академічну різницю* замість *академічної заборгованості*, і модель згенерувала відповідь за ним.
   * *Як виправити:* додати Cross-Encoder переранжування, яке точніше розрізняє близькі юридичні терміни.
4. **Питання [12] («З яких елементів складається навчальний час студента (кредити ЄКТС)?»)**
   * *Тип відмови:* неповна відповідь (розрив контексту на межі чанка).
   * *Причина:* модель назвала академічну годину (40 хв) і пару (80 хв), але пропустила визначення кредиту ЄКТС (30 годин), яке опинилося в сусідньому чанку.
   * *Як виправити:* збільшити `chunk_overlap` або використати стратегію *Parent Document Retriever* (шукати по малих чанках, а в LLM передавати більший батьківський розділ).
5. **Питання [01], [06], [15] (низький ранг релевантного чанка `[5]` у гібридному пошуку)**
   * *Тип відмови:* деградація ранжування (шум BM25).
   * *Причина:* правильний фрагмент опинився лише на 5-й позиції, що знизило `MRR@5` гібридного пошуку нижче за чистий FAISS.
   * *Як виправити:* очистити токенізацію BM25 від українських стоп-слів та канцеляризмів або зменшити вагу BM25 у формулі RRF.

---

### 7. Висновок
Побудовано повноцінну RAG-систему над корпусом нормативних документів університету з веб-інтерфейсом на базі **Gradio**. Система показала високу точність пошуку (`Hit Rate@5 = 100%`), коректно цитує джерела у 95% випадків і має 100% точність відмов на запитання поза базою знань. Основні напрями подальшого покращення — додавання reranker-моделі та очищення лексичного індексу від стоп-слів.